# Weather-related flight disruption prediction with AutoGluon

Predict a binary flight outcome from flight schedules and weather observations. The default target is `weather_disruption`.

| Split | Departure period (UTC) | Use |
| --- | --- | --- |
| Training | 2019–2022 | Fit models |
| Validation | 2023 | Select models |
| Calibration | 2024 | Select decision threshold |
| Test | 2025 (available records) | Final evaluation |

## 1. Select airports

Set `ORIGIN_AIRPORTS` to one or more IATA code. Name each experiment using `EXPERIMENT_NAME`. Date ranges use UTC and exclude the end date.

Results and models are saved under `data/experiments/<EXPERIMENT_NAME>/`. Set `scratch = True` to delete and recreate this experiment directory when the setup cell runs.

In [ ]:
from pathlib import Path
import json
import shutil
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from IPython.display import display
from autogluon.tabular import TabularDataset, TabularPredictor


ORIGIN_AIRPORTS = ['DFW']  # E.g. ['DFW', 'ORD']; None for all airports.
EXPERIMENT_NAME = '1_airport'
scratch = False  # True clears the selected experiment directory.

In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Start in the project root or notebooks directory.'

EXPERIMENT_DIR = ROOT / 'data' / 'experiments' / EXPERIMENT_NAME
MODEL_DIR = EXPERIMENT_DIR / 'model'
RESULTS_PATH = EXPERIMENT_DIR / 'results.json'
if scratch:
    # Restrict deletion to a direct child of data/experiments.
    if (EXPERIMENT_DIR.parent != ROOT / 'data' / 'experiments'
            or EXPERIMENT_DIR.resolve() != EXPERIMENT_DIR):
        raise ValueError('EXPERIMENT_NAME must identify a directory directly under data/experiments.')
    if EXPERIMENT_DIR.exists():
        shutil.rmtree(EXPERIMENT_DIR)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print('Experiment directory:', EXPERIMENT_DIR)

# Use non-overlapping UTC date ranges with exclusive end dates.
PERIODS = {
    'training': ('2019-01-01', '2023-01-01'),
    'validation': ('2023-01-01', '2024-01-01'),
    'calibration': ('2024-01-01', '2025-01-01'),
    'test': ('2025-01-01', '2026-01-01'),
}

previous_end = None
for name, (start, end) in PERIODS.items():
    assert pd.Timestamp(start) < pd.Timestamp(end), f'{name}: start must precede end.'
    if previous_end is not None:
        assert pd.Timestamp(previous_end) <= pd.Timestamp(start), 'Periods must not overlap.'
    previous_end = end

DATA = ROOT / 'data' / 'bts_flights_weather' / 'flights_weather'
files = sorted(DATA.glob('year=*/*.parquet'))
assert files, 'Download the BTS flight/weather year partitions first.'
print('Origin airports:', ORIGIN_AIRPORTS if ORIGIN_AIRPORTS is not None else 'all origins')
print('Available year folders:', sorted({file.parent.name for file in files}))

Experiment directory: /root/projects/flight-delay-ml/data/1_airport
Origin airports: ['DFW']
Available year folders: ['year=2019', 'year=2020', 'year=2021', 'year=2022', 'year=2023', 'year=2024', 'year=2025']


## 2. Define the target and its exclusions

To change the binary target, edit the next cell. Set `TARGET` to the label name, list every raw column used to compute it in `TARGET_INPUTS`, and update `build_target` to return one 0/1 value per flight. Choose how missing source values should affect the label in that function.

The default label is 1 for a weather-related cancellation or when both reported weather delay and departure delay exceed 15 minutes.

In [ ]:
TARGET = 'weather_disruption'
# Declare every raw column used by build_target.
TARGET_INPUTS = ['reported_weather_delay_min', 'dep_delay_min', 'cancelled', 'cancellation_reason']
TARGET_EXCLUSIONS = []  # Other columns that reveal the target.

def build_target(inputs):
    delayed_by_weather = (
        inputs['reported_weather_delay_min'].gt(15) & inputs['dep_delay_min'].gt(15)
    )
    cancelled_by_weather = (
        inputs['cancelled'].fillna(False) & inputs['cancellation_reason'].eq('Weather')
    )
    # Treat an unknown outcome as negative for this target.
    return (delayed_by_weather | cancelled_by_weather).fillna(False)

In [4]:
# Approved predictor configuration

GROUPS = ['wx_origin', 'wx_origin_m1', 'wx_origin_m3', 'wx_origin_m6', 'wx_dest', 'wx_dest_m3']
# Allow only fields available at prediction time.
SCHEDULE_FEATURES = ['operating_carrier', 'origin_iata', 'dest_iata', 'distance_miles', 'crs_elapsed_min', 'day_of_week', 'day_of_year', 'month', 'is_us_federal_holiday', 'days_to_nearest_holiday', 'is_holiday_travel_window']
WEATHER_FEATURES = ['wx_origin_temp_c', 'wx_origin_dewpoint_c', 'wx_origin_relative_humidity_pct', 'wx_origin_wind_speed_ms', 'wx_origin_wind_gust_ms', 'wx_origin_wind_dir_deg', 'wx_origin_visibility_m', 'wx_origin_ceiling_m', 'wx_origin_ceiling_unlimited', 'wx_origin_sea_level_pressure_hpa', 'wx_origin_precip_1h_mm', 'wx_origin_snow_depth_cm', 'wx_origin_sky_cover_oktas', 'wx_origin_thunderstorm', 'wx_origin_snow', 'wx_origin_freezing_precip', 'wx_origin_fog', 'wx_origin_rain', 'wx_origin_flight_category', 'wx_origin_flight_category_ordinal', 'wx_origin_obs_missing', 'wx_origin_m1_temp_c', 'wx_origin_m1_wind_speed_ms', 'wx_origin_m1_wind_gust_ms', 'wx_origin_m1_visibility_m', 'wx_origin_m1_ceiling_m', 'wx_origin_m1_precip_1h_mm', 'wx_origin_m1_flight_category_ordinal', 'wx_origin_m1_thunderstorm', 'wx_origin_m1_snow', 'wx_origin_m1_freezing_precip', 'wx_origin_m1_fog', 'wx_origin_m1_rain', 'wx_origin_m1_obs_missing', 'wx_origin_m3_temp_c', 'wx_origin_m3_wind_speed_ms', 'wx_origin_m3_wind_gust_ms', 'wx_origin_m3_visibility_m', 'wx_origin_m3_ceiling_m', 'wx_origin_m3_precip_1h_mm', 'wx_origin_m3_flight_category_ordinal', 'wx_origin_m3_thunderstorm', 'wx_origin_m3_snow', 'wx_origin_m3_freezing_precip', 'wx_origin_m3_fog', 'wx_origin_m3_rain', 'wx_origin_m3_obs_missing', 'wx_origin_m6_temp_c', 'wx_origin_m6_wind_speed_ms', 'wx_origin_m6_wind_gust_ms', 'wx_origin_m6_visibility_m', 'wx_origin_m6_ceiling_m', 'wx_origin_m6_precip_1h_mm', 'wx_origin_m6_flight_category_ordinal', 'wx_origin_m6_thunderstorm', 'wx_origin_m6_snow', 'wx_origin_m6_freezing_precip', 'wx_origin_m6_fog', 'wx_origin_m6_rain', 'wx_origin_m6_obs_missing', 'wx_dest_temp_c', 'wx_dest_dewpoint_c', 'wx_dest_relative_humidity_pct', 'wx_dest_wind_speed_ms', 'wx_dest_wind_gust_ms', 'wx_dest_wind_dir_deg', 'wx_dest_visibility_m', 'wx_dest_ceiling_m', 'wx_dest_ceiling_unlimited', 'wx_dest_sea_level_pressure_hpa', 'wx_dest_precip_1h_mm', 'wx_dest_snow_depth_cm', 'wx_dest_sky_cover_oktas', 'wx_dest_thunderstorm', 'wx_dest_snow', 'wx_dest_freezing_precip', 'wx_dest_fog', 'wx_dest_rain', 'wx_dest_flight_category', 'wx_dest_flight_category_ordinal', 'wx_dest_obs_missing', 'wx_dest_m3_temp_c', 'wx_dest_m3_wind_speed_ms', 'wx_dest_m3_wind_gust_ms', 'wx_dest_m3_visibility_m', 'wx_dest_m3_ceiling_m', 'wx_dest_m3_precip_1h_mm', 'wx_dest_m3_flight_category_ordinal', 'wx_dest_m3_thunderstorm', 'wx_dest_m3_snow', 'wx_dest_m3_freezing_precip', 'wx_dest_m3_fog', 'wx_dest_m3_rain', 'wx_dest_m3_obs_missing']
# Keep target inputs and related leakage columns out of the model.
excluded_predictors = {TARGET, *TARGET_INPUTS, *TARGET_EXCLUSIONS}
FEATURES = [
    column for column in SCHEDULE_FEATURES + WEATHER_FEATURES
    if column not in excluded_predictors
]
# Load the timestamps used for splitting and weather checks alongside the label inputs.
LOAD_COLUMNS = list(dict.fromkeys(
    FEATURES + TARGET_INPUTS + ['info_cutoff_utc', 'sched_dep_utc']
    + [group + '_obs_time_utc' for group in GROUPS]
))

available_columns = pq.read_schema(files[0]).names
unknown_exclusions = set(TARGET_EXCLUSIONS) - set(available_columns)
if unknown_exclusions:
    raise ValueError(f'Unknown TARGET_EXCLUSIONS columns: {sorted(unknown_exclusions)}')
missing_columns = set(LOAD_COLUMNS) - set(available_columns)
assert not missing_columns, f'Missing required columns: {sorted(missing_columns)}'
EXCLUDED_COLUMNS = sorted(set(available_columns) - set(FEATURES))
print('Predictor columns:', len(FEATURES))
print('Source columns excluded from model inputs:', EXCLUDED_COLUMNS)

def prepare_model_data(frame):
    """Create labels and exclude future weather observations."""
    # The cutoff equals scheduled departure in the supplied data.
    cutoff = frame['info_cutoff_utc']
    assert cutoff.notna().all() and cutoff.le(frame['sched_dep_utc']).all(), (
        'Every row needs a cutoff no later than scheduled departure.'
    )
    labels = build_target(frame.loc[:, TARGET_INPUTS])
    if (not isinstance(labels, pd.Series) or not labels.index.equals(frame.index)
            or not labels.isin([0, 1]).all()):
        raise ValueError('build_target must return a 0/1 Series aligned with the input rows.')
    # Mask weather with a missing timestamp or an observation after the cutoff.
    for group in GROUPS:
        stamp = frame[group + '_obs_time_utc']
        invalid = stamp.isna() | stamp.gt(cutoff)
        # Match each field to its group, including more specific prefixes such as wx_origin_m1.
        fields = [column for column in WEATHER_FEATURES
                  if column in FEATURES and column.startswith(group + '_') and not any(
                      column.startswith(other + '_') for other in GROUPS
                      if other.startswith(group + '_'))]
        frame.loc[invalid, fields] = pd.NA
    # Convert missing Arrow values to np.nan for AutoGluon.
    categorical_features = ['operating_carrier', 'origin_iata', 'dest_iata',
                            'wx_origin_flight_category', 'wx_dest_flight_category']
    result = {}
    for column in FEATURES:
        if column in categorical_features:
            values = frame[column].astype(object)
            result[column] = values.where(pd.notna(values), np.nan)
        else:
            result[column] = pd.to_numeric(frame[column], errors='raise').to_numpy(
                dtype=np.float32, na_value=np.nan
            )
    result[TARGET] = labels.to_numpy(dtype=np.int8)
    return pd.DataFrame(result, index=frame.index)


Predictor columns: 105
Source columns excluded from model inputs: ['actual_arr_utc', 'actual_dep_utc', 'actual_elapsed_min', 'air_time_min', 'arr_del15', 'arr_delay_min', 'cancellation_code', 'cancellation_reason', 'cancelled', 'dep_delay_min', 'dest_sched_arrivals_this_hour', 'diverted', 'dst_edge_case', 'flight_date_local_origin', 'flight_number', 'info_cutoff_utc', 'marketing_carrier', 'marketing_carrier_name', 'operating_carrier_name', 'oracle_dest_arr_ceiling_m', 'oracle_dest_arr_flight_category_ordinal', 'oracle_dest_arr_fog', 'oracle_dest_arr_freezing_precip', 'oracle_dest_arr_is_post_cutoff', 'oracle_dest_arr_is_post_event', 'oracle_dest_arr_obs_minus_event_min', 'oracle_dest_arr_obs_missing', 'oracle_dest_arr_obs_time_utc', 'oracle_dest_arr_precip_1h_mm', 'oracle_dest_arr_rain', 'oracle_dest_arr_snow', 'oracle_dest_arr_speci_count_1h', 'oracle_dest_arr_temp_c', 'oracle_dest_arr_thunderstorm', 'oracle_dest_arr_visibility_m', 'oracle_dest_arr_wind_gust_ms', 'oracle_dest_arr_wind

## 3. Load and partition the data

Load the selected flights into a single pandas DataFrame, then split by scheduled departure in UTC. Each split contains the approved predictors and `TARGET`.

In [5]:
# Read only the required columns and selected airports.
airport_filters = None if ORIGIN_AIRPORTS is None else [('origin_iata', 'in', ORIGIN_AIRPORTS)]
flights = pd.read_parquet(
    files, engine='pyarrow', columns=LOAD_COLUMNS, filters=airport_filters,
    dtype_backend='pyarrow', partitioning=None,
)

def split_period(frame, start, end):
    # Include the start date and exclude the end date.
    departure = frame['sched_dep_utc']
    in_period = departure.ge(pd.Timestamp(start, tz='UTC')) & departure.lt(pd.Timestamp(end, tz='UTC'))
    partition = frame.loc[in_period].sort_values('sched_dep_utc', ignore_index=True)
    return prepare_model_data(partition)

train_data = split_period(flights, *PERIODS['training'])
validation_data = split_period(flights, *PERIODS['validation'])
calibration_data = split_period(flights, *PERIODS['calibration'])
test_data = split_period(flights, *PERIODS['test'])

## 4. Train AutoGluon

In [ ]:
# TODO: Convert all four splits to AutoGluon TabularDataset objects.

In [ ]:
# Docs: https://auto.gluon.ai/stable/tutorials/tabular/tabular-quick-start.html
# API: https://auto.gluon.ai/stable/api/autogluon.tabular.TabularPredictor.fit.html
# Models: https://auto.gluon.ai/stable/api/autogluon.tabular.models.html

# TODO: Fit the predictor below, using average_precision as metric to optimize.
# Use validation_data as tuning_data
# IMPORTANT: start training only one model family e.g. included_model_types=['XGB']; When the training works you can extend it with RF, LR, etc. 
# IMPORTANT: use the parameter "time_limit" to control the duration of the experiment.

predictor = TabularPredictor(
    label=TARGET,
    # .....
    # .....
    # .....
    path=str(MODEL_DIR),
)
# TODO: predictor.fit(...)

## 5. Compare models

Review the validation leaderboard. Higher `score_val` (average precision) is better.

In [ ]:
# TODO: Display the validation leaderboard.

## 6. Calibrate the decision threshold (2024)

Choose the classification threshold using the calibration split only.

In [ ]:
# TODO: Calibrate the threshold with calibration_data
# https://auto.gluon.ai/stable/api/autogluon.tabular.TabularPredictor.calibrate_decision_threshold.html

## 7. Evaluate on the test split (2025)

Report final metrics with the threshold chosen on 2024; do not tune on test data.

In [ ]:
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_score, recall_score, f1_score,
                             accuracy_score, confusion_matrix)
y_test = test_data[TARGET]
x_test = test_data.drop(columns=TARGET)
# Use probabilities for ranking metrics and thresholded predictions for classification metrics.
test_probabilities = predictor.predict_proba(x_test, as_multiclass=False)
test_predictions = predictor.predict(x_test)
assert np.isfinite(test_probabilities).all()
test_metrics = pd.Series({
    'flights': len(y_test),
    'positives': int(y_test.sum()),
    'positive_rate': float(y_test.mean()),
    'decision_threshold': predictor.decision_threshold,
    'average_precision': average_precision_score(y_test, test_probabilities),
    'roc_auc': roc_auc_score(y_test, test_probabilities),
    'precision': precision_score(y_test, test_predictions, zero_division=0),
    'recall': recall_score(y_test, test_predictions, zero_division=0),
    'f1': f1_score(y_test, test_predictions, zero_division=0),
    'accuracy': accuracy_score(y_test, test_predictions),
    'alert_rate': float((test_predictions == 1).mean()),
}, name='test')
test_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=[0, 1]),
    index=['actual_negative', 'actual_positive'],
    columns=['predicted_negative', 'predicted_positive'],
)
print('Predictor:', predictor.path)
print('Model:', predictor.model_best)
print(test_metrics.to_string())
print(test_confusion_matrix.to_string())


In [ ]:
# Persist the calibrated predictor and final experiment results.
predictor.save()
experiment_results = {
    'experiment_name': EXPERIMENT_NAME,
    'origin_airports': ORIGIN_AIRPORTS,
    'periods': PERIODS,
    'target': TARGET,
    'target_inputs': TARGET_INPUTS,
    'target_exclusions': TARGET_EXCLUSIONS,
    'features': FEATURES,
    'model_path': str(MODEL_DIR.relative_to(ROOT)),
    'best_model': predictor.model_best,
    'test_metrics': {
        key: int(value) if key in {'flights', 'positives'} else float(value)
        for key, value in test_metrics.items()
    },
    'test_confusion_matrix': test_confusion_matrix.to_dict(orient='index'),
    'validation_leaderboard': json.loads(
        predictor.leaderboard(silent=True).to_json(orient='records')
    ),
}
RESULTS_PATH.write_text(
    json.dumps(experiment_results, indent=2, allow_nan=False) + '\n',
    encoding='utf-8',
)
print('Saved results:', RESULTS_PATH)
print('Saved AutoGluon predictor:', MODEL_DIR)


## 8. Inspect feature importance

Compute [`predictor.feature_importance`](https://auto.gluon.ai/stable/api/autogluon.tabular.TabularPredictor.feature_importance.html) and plot the top 20 features.

In [ ]:
# TODO: Compute feature_importance before plotting.
feature_importance = None

import matplotlib.pyplot as plt
# Select the top 20 features and order the bars by importance.
plot_data = feature_importance.head(20).sort_values('importance')
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_data.index, plot_data['importance'])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Decrease in test average precision after permutation')
ax.set_title('Top 20 feature importances — test partition')
fig.tight_layout()
plt.show()


## Next steps

Using the **airport ranking from the previous notebook**, repeat this analysis for the top **5, 10, 15, and 20 airports** impacted by departure delay / cancellations. Compare the test metrics across the four groups. Change experiment_name each time you try a new combintion.
Feel free to explore, try other combinations and do your own experiments.

GOAL: Identify the combination that maximizes recall metric